<a href="https://colab.research.google.com/github/UAV-Search-and-Rescue/UAV_SAR/blob/main/notebooks/02_rgb_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E0: Frozen-Split RGB-Only YOLOv8s Baseline

This notebook is the executable E0 experiment only. It uses the frozen recording-level split and the validated frame-level RGB pair manifest. It never creates a new split and keeps the two final test contexts out of development cross-validation, training, model selection, and early stopping.

The ZIP is read without extracting unrelated members. Only validated RGB image/annotation pairs referenced by `rgb_dataset_manifest.csv` are materialized into the local YOLO workspace.

In [26]:
from __future__ import annotations

import csv
import os
import shutil
import sys
import zipfile
from collections import Counter
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/UAV_SAR")

SPLIT_MANIFEST = Path("/content/recording_split_manifest.csv")

PAIR_MANIFEST = Path(
    "/content/UAV_SAR/results/rgb_baseline/rgb_dataset_manifest.csv"
)
ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
E0_ROOT = PROJECT_ROOT / "results" / "rgb_baseline" / "e0_yolo_workspace"
RUN_TRAINING = True
EXPECTED_PAIRS = 10_703
EXPECTED_DEVELOPMENT_PAIRS = 9_096
EXPECTED_TEST_PAIRS = 1_607
TEST_CONTEXTS = {"210327_Airfield_FLIR", "210812_Hannegan_Enterprise"}
DEVELOPMENT_CONTEXTS = {
    "210417_MtErie_Enterprise", "210529_Carnation_Enterprise",
    "210924_FHL_Enterprise", "220109_Baker_Enterprise",
}

if not SPLIT_MANIFEST.is_file():
    raise FileNotFoundError(SPLIT_MANIFEST)
if not PAIR_MANIFEST.is_file():
    raise FileNotFoundError(PAIR_MANIFEST)
if not ZIP_PATH.is_file():
    raise FileNotFoundError(ZIP_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"ZIP: {ZIP_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/UAV_SAR
ZIP: /content/drive/MyDrive/WiSARDv1.zip


In [2]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/UAV_SAR")

# Create required directory
split_dir = PROJECT_ROOT / "results" / "dataset_split"
split_dir.mkdir(parents=True, exist_ok=True)

# Copy frozen split into the location expected by the preparation script
src = Path("/content/recording_split_manifest.csv")
dst = split_dir / "recording_split_manifest.csv"

if not src.exists():
    raise FileNotFoundError(src)

shutil.copy2(src, dst)

print("Prepared:")
print(dst)
print("Exists:", dst.exists())

Prepared:
/content/UAV_SAR/results/dataset_split/recording_split_manifest.csv
Exists: True


In [3]:
!ls -lh /content/UAV_SAR/results/dataset_split/

total 8.0K
-rw-r--r-- 1 root root 4.9K Sep  8 17:56 recording_split_manifest.csv


In [4]:
from pathlib import Path

print("=== Required inputs ===")

files = {
    "Split manifest":
        Path("/content/recording_split_manifest.csv"),

    "WiSARD ZIP":
        Path("/content/drive/MyDrive/WiSARDv1.zip"),

    "Preparation script":
        Path("/content/prepare_rgb_baseline.py"),

    "RGB pair manifest":
        Path("/content/UAV_SAR/results/rgb_baseline/rgb_dataset_manifest.csv"),
}

for name, path in files.items():
    print(f"{name:20} {path.exists()}  {path}")

=== Required inputs ===
Split manifest       True  /content/recording_split_manifest.csv
WiSARD ZIP           True  /content/drive/MyDrive/WiSARDv1.zip
Preparation script   True  /content/prepare_rgb_baseline.py
RGB pair manifest    False  /content/UAV_SAR/results/rgb_baseline/rgb_dataset_manifest.csv


In [5]:
PROJECT_ROOT = Path("/content/UAV_SAR")
DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")

In [6]:
!grep -n "PROJECT_ROOT\|DEFAULT_ZIP_PATH" /content/prepare_rgb_baseline.py

17:PROJECT_ROOT = Path("/content/UAV_SAR")
18:SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
19:OUTPUT_DIR = PROJECT_ROOT / "results" / "rgb_baseline"
20:DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
21:LOCAL_ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "WiSARD" / "WiSARDv1.zip"
32:        return DEFAULT_ZIP_PATH


In [7]:
!python /content/prepare_rgb_baseline.py

WiSARD RGB-only E0 dataset preparation
Frozen split manifest: /content/UAV_SAR/results/dataset_split/recording_split_manifest.csv
ZIP inspected read-only: /content/drive/MyDrive/WiSARDv1.zip
VIS recordings: 21
Fully annotated recordings: 14
Partially annotated recordings: 1
Unannotated recordings: 6
Matched image/annotation pairs: 10703
Excluded unannotated images: 5748
Invalid bounding boxes excluded: 13
Images excluded because annotations contain invalid boxes: 13
Malformed annotations: 0
Invalid bounding boxes: 13
Annotation read errors: 0
Class ID distribution: {0: 23808}

Counts by partition
-------------------
development: recordings=15, images=9096, annotations=9096
test: recordings=6, images=1607, annotations=1607

RGB dataset READY FOR E0 TRAINING: YES
No model was trained. The frozen split assignment was not changed.
Images were not opened or decoded; only ZIP paths and annotation text were inspected.
Unannotated recordings remain in their frozen partitions but are excluded f

In [8]:
!grep -n "rgb_dataset_manifest\|UAV_SAR\|WiSARDv1\|recording_split_manifest" \
    /content/prepare_rgb_baseline.py

17:PROJECT_ROOT = Path("/content/UAV_SAR")
18:SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
20:DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
21:LOCAL_ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "WiSARD" / "WiSARDv1.zip"
94:        raise FileNotFoundError(f"WiSARDv1.zip not found at {zip_path}. Mount Google Drive in Colab or set WISAR_ZIP_PATH.")
169:    write_csv(OUTPUT_DIR / "rgb_dataset_manifest.csv", fields, manifest_rows)


In [9]:
from pathlib import Path

script = Path("/content/prepare_rgb_baseline.py")

text = script.read_text()

old = 'PROJECT_ROOT = Path(__file__).resolve().parents[1]'
new = 'PROJECT_ROOT = Path("/content/UAV_SAR")'

if old not in text:
    print("Old PROJECT_ROOT line was not found.")
else:
    text = text.replace(old, new)
    script.write_text(text)
    print("✅ PROJECT_ROOT fixed.")

Old PROJECT_ROOT line was not found.


In [10]:
from pathlib import Path

script = Path("/content/prepare_rgb_baseline.py")
text = script.read_text()

text = text.replace(
    'DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARD") / "WiSARDv1.zip"',
    'DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")'
)

script.write_text(text)

print("✅ ZIP path fixed")

✅ ZIP path fixed


In [11]:
!head -25 /content/prepare_rgb_baseline.py

"""Prepare a supervised RGB index from the frozen context split.

The ZIP is inspected read-only. Image members are never opened or decoded;
only matched annotation members are read as text.
"""

from __future__ import annotations

import csv
import math
import os
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

PROJECT_ROOT = Path("/content/UAV_SAR")
SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "rgb_baseline"
DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
LOCAL_ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "WiSARD" / "WiSARDv1.zip"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
ANNOTATION_EXTENSIONS = {".txt", ".ann", ".label", ".labels"}
REQUIRED_SPLIT_COLUMNS = {"recording_name", "recording_path", "collection_context", "modality", "partition"}



In [12]:
!sed -n '1,110p' /content/prepare_rgb_baseline.py

"""Prepare a supervised RGB index from the frozen context split.

The ZIP is inspected read-only. Image members are never opened or decoded;
only matched annotation members are read as text.
"""

from __future__ import annotations

import csv
import math
import os
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

PROJECT_ROOT = Path("/content/UAV_SAR")
SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "rgb_baseline"
DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
LOCAL_ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "WiSARD" / "WiSARDv1.zip"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
ANNOTATION_EXTENSIONS = {".txt", ".ann", ".label", ".labels"}
REQUIRED_SPLIT_COLUMNS = {"recording_name", "recording_path", "collection_context", "modality", "partition"}


def resolve_zip_path() -> Path:
    configured = os.environ.

In [13]:
from pathlib import Path

print("=== /content ===")
!ls -lah /content

print("\n=== UAV_SAR ===")
!ls -lah /content/UAV_SAR 2>/dev/null || echo "UAV_SAR does NOT exist"

print("\n=== results ===")
!ls -lah /content/UAV_SAR/results 2>/dev/null || echo "results does NOT exist"

print("\n=== rgb_baseline ===")
!ls -lah /content/UAV_SAR/results/rgb_baseline 2>/dev/null || echo "rgb_baseline does NOT exist"

=== /content ===
total 44K
drwxr-xr-x 1 root root 4.0K Sep  8 18:07 .
drwxr-xr-x 1 root root 4.0K Sep  8 17:55 ..
drwxr-xr-x 4 root root 4.0K Aug 24 13:27 .config
drwx------ 5 root root 4.0K Sep  8 18:03 drive
-rw-r--r-- 1 root root 9.8K Sep  8 18:13 prepare_rgb_baseline.py
-rw-r--r-- 1 root root 4.9K Sep  8 17:56 recording_split_manifest.csv
drwxr-xr-x 1 root root 4.0K Aug 24 13:28 sample_data
drwxr-xr-x 3 root root 4.0K Sep  8 18:07 UAV_SAR

=== UAV_SAR ===
total 12K
drwxr-xr-x 3 root root 4.0K Sep  8 18:07 .
drwxr-xr-x 1 root root 4.0K Sep  8 18:07 ..
drwxr-xr-x 4 root root 4.0K Sep  8 18:13 results

=== results ===
total 16K
drwxr-xr-x 4 root root 4.0K Sep  8 18:13 .
drwxr-xr-x 3 root root 4.0K Sep  8 18:07 ..
drwxr-xr-x 2 root root 4.0K Sep  8 18:07 dataset_split
drwxr-xr-x 2 root root 4.0K Sep  8 18:13 rgb_baseline

=== rgb_baseline ===
total 2.6M
drwxr-xr-x 2 root root 4.0K Sep  8 18:13 .
drwxr-xr-x 4 root root 4.0K Sep  8 18:13 ..
-rw-r--r-- 1 root root 2.5M Sep  8 18:13 rgb_da

In [14]:
print("\n=== WiSARD folder ===")
!ls -lah "/content/drive/MyDrive/WiSARD" 2>/dev/null || echo "WiSARD folder NOT found"

print("\n=== WiSARD ZIP ===")
!ls -lh "/content/drive/MyDrive/WiSARD/WiSARDv1.zip" 2>/dev/null || echo "WiSARDv1.zip NOT found"


=== WiSARD folder ===
WiSARD folder NOT found

=== WiSARD ZIP ===
WiSARDv1.zip NOT found


In [15]:
from pathlib import Path

print("=== MyDrive ===")
!ls -lah "/content/drive/MyDrive"

print("\n=== Searching entire mounted Drive ===")
!find "/content/drive" -type f -iname "*.zip" 2>/dev/null | head -50

=== MyDrive ===
total 43G
-rw------- 1 root root  8.8K Sep  3 17:19  10_Months_and_Forever_Bubuuu.zip
-rw------- 1 root root  174K Feb  8  2023  1671528656786.jpg
-rw------- 1 root root  2.5M Mar  9  2023 '639c71228b856f011f4d0b91_##_Unit & Dimension - 02 : Class Notes (PJ44EB).pdf'
-rw------- 1 root root  805K Jun  9  2024  881150125908953_signed.pdf
-rw------- 1 root root   51M Jan 26  2024  aa11c966-1eb3-40df-96e7-f643110fd32c-1705742660113-4102609153252655.pdf
-rw------- 1 root root   13K Mar 20  2022  ACKNOWLEDGEMENT.docx
-rw------- 1 root root   183 Jun 27 13:16 'AI-Powered Digital Public Safety Intelligence Platform - System Specification and Project Blueprint.gdoc'
-rw------- 1 root root   183 Jul 15 02:24 'Assessment Task - AI - QAG Agentic AI Pipeline for CBSE NCERT Question Generation.gdoc'
-rw------- 1 root root  739K Aug 31  2021  car2.jpg
-rw------- 1 root root  3.0K Aug 31  2021 'car 3.jpg'
-rw------- 1 root root  3.3K Aug 31  2021  car.jpg
-rw------- 1 root root   93K M

In [16]:
!find /content -name "rgb_dataset_manifest.csv" -o -name "recording_split_manifest.csv" 2>/dev/null

/content/UAV_SAR/results/dataset_split/recording_split_manifest.csv
/content/UAV_SAR/results/rgb_baseline/rgb_dataset_manifest.csv
/content/recording_split_manifest.csv


In [17]:
!grep -n "PROJECT_ROOT\|DEFAULT_ZIP_PATH" /content/prepare_rgb_baseline.py

17:PROJECT_ROOT = Path("/content/UAV_SAR")
18:SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
19:OUTPUT_DIR = PROJECT_ROOT / "results" / "rgb_baseline"
20:DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")
21:LOCAL_ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "WiSARD" / "WiSARDv1.zip"
32:        return DEFAULT_ZIP_PATH


In [19]:
from pathlib import Path

script = Path("/content/prepare_rgb_baseline.py")
text = script.read_text()

text = text.replace(
    'DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARD") / "WiSARDv1.zip"',
    'DEFAULT_ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")'
)

script.write_text(text)

print("✅ ZIP path fixed.")

✅ ZIP path fixed.


In [20]:
!find /content/drive -name "rgb_dataset_manifest.csv" -o -name "recording_split_manifest.csv" 2>/dev/null

In [21]:
!ls "/content/drive/MyDrive"

 10_Months_and_Forever_Bubuuu.zip
 1671528656786.jpg
'639c71228b856f011f4d0b91_##_Unit & Dimension - 02 : Class Notes (PJ44EB).pdf'
 881150125908953_signed.pdf
 aa11c966-1eb3-40df-96e7-f643110fd32c-1705742660113-4102609153252655.pdf
 ACKNOWLEDGEMENT.docx
'AI-Powered Digital Public Safety Intelligence Platform - System Specification and Project Blueprint.gdoc'
'Assessment Task - AI - QAG Agentic AI Pipeline for CBSE NCERT Question Generation.gdoc'
 car2.jpg
'car 3.jpg'
 car.jpg
'CAR MANAGEMENT.pdf'
'caste certificate chinu.pdf'
 CATEGORY_CERTIFICATE.pdf
 CERTIFICATE_ENG.docx
'class 12 project PIYUSH KUMAR'
'class 12 project PIYUSH KUMAR (1)'
'Colab Notebooks'
'Copy of Crowdgen Working Hours.gsheet'
'cse 21 cse 29.pdf'
'Detailed CV piyush.pdf'
'Document from PIYUSH KUMAR PAL'
 ENG_FRONT_PAGE.docx
 Ephoto360.com_162aaa7b657435.jpg
'file handaling.pdf'
 GDToT
'Google Earth'
 IMG_20210706_133753.jpg
 IMG_20220823_115559.jpg
 IMG_20230309_224923.jpg
 IMG_20230310_190919.jpg
 lv_0_20260110122

In [22]:
from pathlib import Path

print("Script:", Path("/content/prepare_rgb_baseline.py").exists())
print("Split:", Path("/content/recording_split_manifest.csv").exists())
print("ZIP:", Path("/content/drive/MyDrive/WiSARD/WiSARDv1.zip").exists())

Script: True
Split: True
ZIP: False


In [ ]:
from pathlib import Path

drive_root = Path("/content/drive")

print("Drive mounted:", drive_root.exists())
print("\nMyDrive contents:")
!ls -lah "/content/drive/MyDrive"

print("\nSearching for WiSARDv1.zip...")
!find "/content/drive" -type f -iname "WiSARDv1.zip" 2>/dev/null

Drive mounted: True

MyDrive contents:
total 43G
-rw------- 1 root root  8.8K Sep  3 17:19  10_Months_and_Forever_Bubuuu.zip
-rw------- 1 root root  174K Feb  8  2023  1671528656786.jpg
-rw------- 1 root root  2.5M Mar  9  2023 '639c71228b856f011f4d0b91_##_Unit & Dimension - 02 : Class Notes (PJ44EB).pdf'
-rw------- 1 root root  805K Jun  9  2024  881150125908953_signed.pdf
-rw------- 1 root root   51M Jan 26  2024  aa11c966-1eb3-40df-96e7-f643110fd32c-1705742660113-4102609153252655.pdf
-rw------- 1 root root   13K Mar 20  2022  ACKNOWLEDGEMENT.docx
-rw------- 1 root root   183 Jun 27 13:16 'AI-Powered Digital Public Safety Intelligence Platform - System Specification and Project Blueprint.gdoc'
-rw------- 1 root root   183 Jul 15 02:24 'Assessment Task - AI - QAG Agentic AI Pipeline for CBSE NCERT Question Generation.gdoc'
-rw------- 1 root root  739K Aug 31  2021  car2.jpg
-rw------- 1 root root  3.0K Aug 31  2021 'car 3.jpg'
-rw------- 1 root root  3.3K Aug 31  2021  car.jpg
-rw----

In [ ]:
!find "/content/drive" -type f -iname "WiSARDv1.zip" 2>/dev/null

/content/drive/MyDrive/WiSARDv1.zip


In [23]:
import csv

with open(dst, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Columns:", reader.fieldnames)
    rows = list(reader)
    print("Rows:", len(rows))

Columns: ['recording_name', 'recording_path', 'collection_context', 'modality', 'image_count', 'partition', 'cv_fold']
Rows: 47


In [24]:
from collections import Counter
import csv

# Exact frozen-manifest schema.
FROZEN_COLUMNS = [
    "recording_name",
    "recording_path",
    "collection_context",
    "modality",
    "image_count",
    "partition",
    "cv_fold",
]

# Exact frame-level RGB-manifest schema produced by prepare_rgb_baseline.py.
PAIR_COLUMNS = [
    "image_member_path",
    "annotation_member_path",
    "recording_name",
    "recording_path",
    "collection_context",
    "partition",
    "modality",
]

# ------------------------------------------------------------------
# Load the two different manifests.
# ------------------------------------------------------------------

with SPLIT_MANIFEST.open(newline="", encoding="utf-8") as handle:
    frozen_rows = list(csv.DictReader(handle))

with PAIR_MANIFEST.open(newline="", encoding="utf-8") as handle:
    pair_rows = list(csv.DictReader(handle))

# ------------------------------------------------------------------
# Validate manifest schemas.
# ------------------------------------------------------------------

if not frozen_rows:
    raise ValueError("Frozen recording split manifest is empty.")

if not pair_rows:
    raise ValueError("RGB frame-level manifest is empty.")

if set(frozen_rows[0].keys()) != set(FROZEN_COLUMNS):
    raise ValueError(
        "Frozen manifest columns do not match the frozen E0 contract.\n"
        f"Expected: {FROZEN_COLUMNS}\n"
        f"Found: {list(frozen_rows[0].keys())}"
    )

if set(pair_rows[0].keys()) != set(PAIR_COLUMNS):
    raise ValueError(
        "RGB pair manifest columns do not match the frozen E0 contract.\n"
        f"Expected: {PAIR_COLUMNS}\n"
        f"Found: {list(pair_rows[0].keys())}"
    )

# ------------------------------------------------------------------
# Validate RGB pair counts.
# ------------------------------------------------------------------

if len(pair_rows) != EXPECTED_PAIRS:
    raise ValueError(
        f"Expected {EXPECTED_PAIRS:,} RGB pairs, found {len(pair_rows):,}"
    )

partition_counts = Counter(row["partition"] for row in pair_rows)

expected_partition_counts = Counter({
    "development": EXPECTED_DEVELOPMENT_PAIRS,
    "test": EXPECTED_TEST_PAIRS,
})

if partition_counts != expected_partition_counts:
    raise ValueError(
        "RGB pair partition counts do not match the frozen E0 counts.\n"
        f"Expected: {dict(expected_partition_counts)}\n"
        f"Found: {dict(partition_counts)}"
    )

# ------------------------------------------------------------------
# Validate modality.
# ------------------------------------------------------------------

if any(row["modality"] != "VIS" for row in pair_rows):
    raise ValueError("The RGB pair manifest contains a non-VIS row.")

# ------------------------------------------------------------------
# Validate every RGB pair against the frozen recording-level split.
# ------------------------------------------------------------------

frozen_by_recording = {
    row["recording_name"]: row
    for row in frozen_rows
}

for row in pair_rows:
    recording_name = row["recording_name"]

    frozen = frozen_by_recording.get(recording_name)

    if frozen is None:
        raise ValueError(
            f"Pair recording is missing from the frozen recording split: "
            f"{recording_name}"
        )

    for field in (
        "recording_path",
        "collection_context",
        "partition",
        "modality",
    ):
        if row[field] != frozen[field]:
            raise ValueError(
                f"Frozen assignment mismatch for {recording_name} "
                f"in field '{field}': "
                f"pair='{row[field]}' vs frozen='{frozen[field]}'"
            )

# ------------------------------------------------------------------
# Validate context membership.
#
# A frozen context may legitimately have zero supervised RGB pairs.
# Therefore we require pair contexts to be a SUBSET of the frozen
# development/test contexts, rather than requiring every context to
# appear in the RGB frame manifest.
# ------------------------------------------------------------------

pair_contexts = {
    row["collection_context"]
    for row in pair_rows
}

allowed_contexts = TEST_CONTEXTS | DEVELOPMENT_CONTEXTS

unexpected_contexts = pair_contexts - allowed_contexts

if unexpected_contexts:
    raise ValueError(
        f"Unexpected E0 contexts: {sorted(unexpected_contexts)}"
    )

# Development and test context definitions themselves must be disjoint.
overlap = TEST_CONTEXTS & DEVELOPMENT_CONTEXTS

if overlap:
    raise ValueError(
        f"Test and development contexts overlap: {sorted(overlap)}"
    )

# ------------------------------------------------------------------
# Determine which frozen contexts actually have supervised RGB pairs.
# ------------------------------------------------------------------

test_pair_contexts = {
    row["collection_context"]
    for row in pair_rows
    if row["partition"] == "test"
}

development_pair_contexts = {
    row["collection_context"]
    for row in pair_rows
    if row["partition"] == "development"
}

# ------------------------------------------------------------------
# Report the validated RGB dataset.
# ------------------------------------------------------------------

development_pairs = sum(
    row["partition"] == "development"
    for row in pair_rows
)

test_pairs = sum(
    row["partition"] == "test"
    for row in pair_rows
)

print(f"RGB pairs: {len(pair_rows):,}")
print(f"Development pairs: {development_pairs:,}")
print(f"Test pairs: {test_pairs:,}")

print(
    "Development RGB contexts:",
    sorted(development_pair_contexts)
)

print(
    "Test RGB contexts with supervised annotations:",
    sorted(test_pair_contexts)
)

# ------------------------------------------------------------------
# Report frozen test contexts that have no supervised RGB pairs.
# This is a NOTE, not an error.
# ------------------------------------------------------------------

missing_test_contexts = TEST_CONTEXTS - test_pair_contexts

if missing_test_contexts:
    print()
    print(
        "NOTE: The following frozen test context(s) have zero "
        "supervised RGB pairs because their VIS recordings are "
        "unannotated. They remain in the frozen test partition "
        "but are excluded from E0 RGB evaluation:"
    )
    print(sorted(missing_test_contexts))

print()
print("Frozen recording-level assignments verified.")
print("No new split created.")
print("E0 RGB manifest validation: PASS")

RGB pairs: 10,703
Development pairs: 9,096
Test pairs: 1,607
Development RGB contexts: ['210417_MtErie_Enterprise', '210529_Carnation_Enterprise', '210924_FHL_Enterprise', '220109_Baker_Enterprise']
Test RGB contexts with supervised annotations: ['210327_Airfield_FLIR']

NOTE: The following frozen test context(s) have zero supervised RGB pairs because their VIS recordings are unannotated. They remain in the frozen test partition but are excluded from E0 RGB evaluation:
['210812_Hannegan_Enterprise']

Frozen recording-level assignments verified.
No new split created.
E0 RGB manifest validation: PASS


In [25]:
def materialize_pairs(rows: list[dict[str, str]], workspace: Path) -> Path:
    if workspace.exists():
        shutil.rmtree(workspace)
    for partition in ("development", "test"):
        (workspace / "images" / partition).mkdir(parents=True, exist_ok=True)
        (workspace / "labels" / partition).mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        member_names = {info.filename.replace("\\", "/") for info in archive.infolist()}
        for row in sorted(rows, key=lambda item: (item["partition"], item["image_member_path"])):
            image_member = row["image_member_path"]
            annotation_member = row["annotation_member_path"]
            if image_member not in member_names or annotation_member not in member_names:
                raise FileNotFoundError(f"Manifest member missing from ZIP: {image_member} / {annotation_member}")
            image_target = workspace / "images" / row["partition"] / Path(image_member).name
            label_target = workspace / "labels" / row["partition"] / f"{Path(image_member).stem}.txt"
            image_target.write_bytes(archive.read(image_member))
            label_target.write_bytes(archive.read(annotation_member))

    with (workspace / "pair_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=PAIR_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    return workspace

E0_WORKSPACE = materialize_pairs(pair_rows, E0_ROOT)

print(f"Materialized validated pairs into {E0_WORKSPACE}")
print("Only manifest-referenced members were extracted; the original ZIP was not modified.")

Materialized validated pairs into /content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace
Only manifest-referenced members were extracted; the original ZIP was not modified.


## Development grouped cross-validation and final test

The four CV folds below are context-level only. Final test contexts are never used in CV or training. After CV, the final YOLOv8s model is trained on all development pairs with validation disabled, then evaluated exactly once on the frozen test pairs.

In [27]:
def write_yolo_data_yaml(path: Path, train_list: Path, val_list: Path | None) -> None:
    lines = [f"train: {train_list.as_posix()}"]
    if val_list is not None:
        lines.append(f"val: {val_list.as_posix()}")
    lines.extend(["nc: 1", "names: ['person']"])
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_image_list(path: Path, rows: list[dict[str, str]]) -> None:
    paths = [str((E0_WORKSPACE / "images" / row["partition"] / Path(row["image_member_path"]).name).resolve()) for row in rows]
    path.write_text("\n".join(paths) + "\n", encoding="utf-8")


def rows_for_contexts(contexts: set[str]) -> list[dict[str, str]]:
    return [row for row in pair_rows if row["collection_context"] in contexts]


In [28]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 5.8 MB/s eta 0:00:00


In [39]:
from pathlib import Path
import pandas as pd

def epochs_completed_from_results(run_root: Path) -> int:
    results_csv = run_root / "results.csv"

    if not results_csv.is_file():
        return 0

    df = pd.read_csv(results_csv)

    if df.empty or "epoch" not in df.columns:
        return 0

    return int(df["epoch"].max()) + 1

In [31]:
from pathlib import Path
import pandas as pd

def epochs_completed_from_results(run_root: Path) -> int:
    """
    Return the highest completed epoch recorded in Ultralytics results.csv.
    Ultralytics records epoch values as zero-based in some contexts, so
    normalize carefully by treating the maximum observed value + 1 as
    the number of completed epochs when necessary.
    """
    results_csv = run_root / "results.csv"

    if not results_csv.is_file():
        return 0

    df = pd.read_csv(results_csv)

    if "epoch" not in df.columns or df.empty:
        return 0

    max_epoch = int(df["epoch"].max())

    # Ultralytics training results use zero-based epoch values.
    return max_epoch + 1

In [32]:
print(E0_ROOT)
print((E0_ROOT / "cv" / "fold_1").exists())
print(list((E0_ROOT / "cv" / "fold_1").rglob("*"))[:20])

/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace
True
[PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/train_images.txt'), PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/data.yaml'), PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/val_images.txt')]


In [34]:
try:
    from ultralytics import YOLO
except ImportError as error:
    raise ImportError(
        "Install ultralytics in Colab before running E0 training: "
        "pip install ultralytics"
    ) from error

TOTAL_EPOCHS = 100
CHUNK_EPOCHS = 10
BATCH_SIZE = 8
SEED = 20260905
PATIENCE = 0

CV_RESULTS = []

for fold_number, validation_context in enumerate(
    sorted(DEVELOPMENT_CONTEXTS), start=1
):
    fold_root = E0_ROOT / "cv" / f"fold_{fold_number}"
    fold_root.mkdir(parents=True, exist_ok=True)

    validation_rows = rows_for_contexts({validation_context})
    training_rows = rows_for_contexts(
        DEVELOPMENT_CONTEXTS - {validation_context}
    )

    train_list = fold_root / "train_images.txt"
    val_list = fold_root / "val_images.txt"
    yaml_path = fold_root / "data.yaml"

    write_image_list(train_list, training_rows)
    write_image_list(val_list, validation_rows)
    write_yolo_data_yaml(yaml_path, train_list, val_list)

    assert not (
        {row["collection_context"] for row in training_rows}
        & {validation_context}
    )
    assert not (
        {row["collection_context"] for row in validation_rows}
        & TEST_CONTEXTS
    )

    # Check whether this fold has already completed all 100 epochs.
    run_root = E0_ROOT / "runs" / f"cv_fold_{fold_number}"
    completed_epochs = epochs_completed_from_results(run_root)

    print(
        f"\nFold {fold_number} | "
        f"Validation context: {validation_context}"
    )
    print(
        f"Completed epochs: {completed_epochs}/{TOTAL_EPOCHS}"
    )

    if completed_epochs >= TOTAL_EPOCHS:
        print("Fold already complete. Skipping training.")
        continue

    remaining_epochs = TOTAL_EPOCHS - completed_epochs
    chunk_epochs = min(CHUNK_EPOCHS, remaining_epochs)

    last_checkpoint = run_root / "weights" / "last.pt"

    if last_checkpoint.exists():
        print(
            f"Resuming from checkpoint: {last_checkpoint}"
        )
        model = YOLO(str(last_checkpoint))
        resume_flag = True
    else:
        print("Starting new fold from yolov8s.pt")
        model = YOLO("yolov8s.pt")
        resume_flag = False

    print(
        f"Running next chunk: {chunk_epochs} epochs "
        f"(target total: {TOTAL_EPOCHS})"
    )

    if RUN_TRAINING:
        model.train(
            data=str(yaml_path),
            project=str(E0_ROOT / "runs"),
            name=f"cv_fold_{fold_number}",
            epochs=TOTAL_EPOCHS,
            imgsz=640,
            batch=BATCH_SIZE,
            seed=SEED,
            patience=PATIENCE,
            val=True,
            resume=resume_flag,
        )

        completed_after = epochs_completed_from_results(fold_root)

        print(
            f"Fold {fold_number}: "
            f"{completed_epochs} → {completed_after} epochs completed."
        )

    CV_RESULTS.append(
        {
            "fold": fold_number,
            "validation_context": validation_context,
            "epochs_completed": epochs_completed_from_results(fold_root),
        }
    )

print(f"\nPrepared {len(DEVELOPMENT_CONTEXTS)} grouped development CV folds.")
print("Final test contexts were excluded from every fold.")


Fold 1 | Validation context: 210417_MtErie_Enterprise
Completed epochs: 0/100
Starting new fold from yolov8s.pt
Running next chunk: 10 epochs (target total: 100)
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kob

Exception ignored in sys.unraisablehookException ignored in atexit callback <function _exit_function at 0x7d49ba8b16c0>: Exception ignored in atexit callback <function _exit_function at 0x7d49ba8b16c0>:
<built-in function unraisablehook>:
Traceback (most recent call last):
Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/util.py", line 405, in _exit_function
  File "/usr/lib/python3.13/multiprocessing/util.py", line 431, in _exit_function
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 561, in write
            _run_finalizers()self.pub_thread.schedule(self._flush)

  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 212, in schedule
    f()  File "/usr/lib/python3.13/multiprocessing/util.py", line 371, in _run_finalizers
_run_finalizers(0)
    finalizer()  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 518, in _flush


      File "/usr/lib

KeyboardInterrupt: 

self.schedule(lambda: self._really_send(*args, **kwargs))
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 235, in _really_send
    ctx, pipe_out = self._setup_pipe_out()
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 156, in _setup_pipe_out
    ctx = zmq.Context()

In [38]:
from pathlib import Path
import pandas as pd

run_root = E0_ROOT / "runs" / "cv_fold_1"

results_csv = run_root / "results.csv"
last_pt = run_root / "weights" / "last.pt"

print("Run directory:", run_root)
print("results.csv exists:", results_csv.exists())
print("last.pt exists:", last_pt.exists())

if results_csv.exists():
    df = pd.read_csv(results_csv)
    print("Last recorded epoch:", df["epoch"].iloc[-1])
    print("Number of recorded rows:", len(df))

Run directory: /content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/runs/cv_fold_1
results.csv exists: True
last.pt exists: True
Last recorded epoch: 20
Number of recorded rows: 20


In [36]:
/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace
True
[PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/train_images.txt'), PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/data.yaml'), PosixPath('/content/UAV_SAR/results/rgb_baseline/e0_yolo_workspace/cv/fold_1/val_images.txt')]


NameError: name 'content' is not defined

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/WiSARDv1.zip")

print("ZIP exists:", ZIP_PATH.is_file())
print("ZIP:", ZIP_PATH)

ZIP exists: True
ZIP: /content/drive/MyDrive/WiSARDv1.zip


In [ ]:
final_root = E0_ROOT / "final"
final_root.mkdir(parents=True, exist_ok=True)
development_rows = rows_for_contexts(DEVELOPMENT_CONTEXTS)
test_rows = rows_for_contexts(TEST_CONTEXTS)
final_train_list = final_root / "development_images.txt"
test_list = final_root / "test_images.txt"
final_yaml = final_root / "development_data.yaml"
test_yaml = final_root / "test_data.yaml"
write_image_list(final_train_list, development_rows)
write_image_list(test_list, test_rows)
write_yolo_data_yaml(final_yaml, final_train_list, None)
test_yaml.write_text(
    f"path: {E0_WORKSPACE.as_posix()}\n"
    f"val: {test_list.as_posix()}\n"
    "names:\n  0: person\n",
    encoding="utf-8",
)

if RUN_TRAINING:
    final_model = YOLO("yolov8s.pt")
    final_model.train(
        data=str(final_yaml),
        project=str(E0_ROOT / "runs"),
        name="final_development",
        epochs=100,
        imgsz=640,
        seed=20260905,
        patience=0,
        val=False,
    )
    IMAGE_SIZE = 640
    BATCH_SIZE = 8
    DEVICE = 0 if __import__("torch").cuda.is_available() else "cpu"
    WORKERS = 2
    test_metrics = final_model.val(
        data=str(test_yaml),
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        split="val",
    )
    print("Final model evaluated once on the frozen test pairs.")
else:
    print("RUN_TRAINING=False: prepared final train/test references without training.")

In [ ]:
print("E0 RGB-only baseline configuration")
print(f"Validated RGB pairs: {len(pair_rows):,}")
print(f"Development pairs: {len(development_rows):,}")
print(f"Final test pairs: {len(test_rows):,}")
print(f"Development contexts: {sorted(DEVELOPMENT_CONTEXTS)}")
print(f"Final test contexts: {sorted(TEST_CONTEXTS)}")
print("No chronological split or alternate manifest was used.")
print("Final test data was excluded from CV and training, then evaluated once.")